# Lesson 12D: GANs - Practical

Train a GAN to generate 2D data distributions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons

class Generator(nn.Module):
    def __init__(self, latent_dim=10, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )
    
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, input_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

# Generate real data
X_real, _ = make_moons(n_samples=1000, noise=0.1, random_state=42)
X_real = torch.FloatTensor(X_real)

# Initialize GAN
G = Generator(latent_dim=10, output_dim=2)
D = Discriminator(input_dim=2)
optimizer_G = optim.Adam(G.parameters(), lr=0.001)
optimizer_D = optim.Adam(D.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Train
for epoch in range(500):
    # Train Discriminator
    z = torch.randn(64, 10)
    fake = G(z)
    real_labels = torch.ones(64, 1)
    fake_labels = torch.zeros(64, 1)
    
    D_real = D(X_real[:64])
    D_fake = D(fake.detach())
    loss_D = criterion(D_real, real_labels) + criterion(D_fake, fake_labels)
    
    optimizer_D.zero_grad()
    loss_D.backward()
    optimizer_D.step()
    
    # Train Generator
    z = torch.randn(64, 10)
    fake = G(z)
    D_fake = D(fake)
    loss_G = criterion(D_fake, real_labels)
    
    optimizer_G.zero_grad()
    loss_G.backward()
    optimizer_G.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/500, D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")

# Generate and visualize
with torch.no_grad():
    z = torch.randn(500, 10)
    generated = G(z).numpy()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_real[:, 0], X_real[:, 1], alpha=0.5, s=10)
plt.title('Real Data (Moons)', fontweight='bold')
plt.subplot(1, 2, 2)
plt.scatter(generated[:, 0], generated[:, 1], alpha=0.5, s=10, c='red')
plt.title('GAN Generated Data', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ GAN successfully learned to generate moon distribution!")